# Nepali LLM Evaluation Dashboard  Llama-3.2-3B
### Base vs. Fine-Tuned

This notebook launches a **Gradio** app with three tabs (Question Answering, Translation, Summarization).
Each comparison now also gets an automatic **LLM-as-Judge verdict** (Gemini 2.5 Flash, free tier) scoring base vs. fine-tuned output.
For whichever task you pick, entering an input and clicking **Run** will:

1. Generate output from the **base** model for **Llama-3.2-3B**.
2. Generate output from the **fine-tuned** model for **Llama-3.2-3B**.
3. Show both outputs side by side on the same page.

This is scoped to **Llama-3.2-3B only**.

**Before running:**
- Kaggle notebook settings → **Accelerator: GPU T4 x2** (or P100), **Internet: On**.
- Model IDs default to the **ungated Unsloth 4-bit mirror** (`unsloth/...`) so you don't need to request Meta's Llama license/gate approval.


In [1]:
# 1. Install dependencies (first run only)
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate peft gradio sentencepiece
# PEFT's LoRA adapter loader eagerly checks for `torchao` even though we never use it, and
# Kaggle's preinstalled torchao (0.10.0) is older than what current PEFT expects (>=0.16.0),
# which raises an ImportError the moment you attach *any* adapter. We don't need torchao at
# all here, so the simplest fix is to remove it rather than fight version pinning.
!pip uninstall -y -q torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 65.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

**Important:** if this is the first time installing/upgrading `bitsandbytes` (or removing `torchao`)
in this Kaggle session, go to **Run → Restart Session** now, then re-run from the top. Kaggle's base
image ships an older `bitsandbytes` and a `torchao` version that's already imported into memory once any
cell runs; changing the installed packages alone doesn't replace what's already loaded, so a restart is
required or you'll hit the same `ImportError`s again even though the pip commands succeeded.


In [2]:
# 2. Imports
import re
from contextlib import nullcontext

import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: False


In [3]:
# 3. Configuration  base model + LoRA adapter per task (Llama-3.2-3B)

CONFIG = {
    "qa": {
        "llama": {"base": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
                  "adapter": "iwasbinod/llama-3.2-3b_qa_v2"},
    },
    "translation": {
        "llama": {"base": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
                  "adapter": "iwasbinod/llama-3.2-3b-nepali-english-translation-with_syn_v4"},
    },
    "summarization": {
        "llama": {"base": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
                  "adapter": "iwasbinod/summarization-llama3.2-3b-real-syn-highquality"},
    },
}

FAMILY_LABEL = {"llama": "Llama-3.2-3B"}

# GEN_PARAMS = {
#     "qa":            {"max_new_tokens": 100,  "temperature": 0.0},
#     "translation":   {"max_new_tokens": 200, "temperature": 0.2},
#     "summarization": {"max_new_tokens": 260, "temperature": 0.3},
# }
GEN_PARAMS = {
    "qa":            {"max_new_tokens": 96,  "temperature": 0.0},
    "translation":   {"max_new_tokens": 220, "temperature": 0.15},
    "summarization": {"max_new_tokens": 280, "temperature": 0.25},
}

In [4]:
# 4. Model loading with adapter caching (base weights loaded once for Llama-3.2-3B),
#    then each task's LoRA adapter is attached as a *named* adapter on that same base model.
#    Base-model behavior is obtained via PEFT's `disable_adapter()` context manager, so we only
#    ever keep 1 full model resident (Llama-3.2-3B), not 3.
#
#    Note: the `unsloth/...-bnb-4bit` repos already embed a quantization_config in their
#    config.json, so we do NOT pass one explicitly (that would just trigger a harmless-but-noisy
#    warning and get ignored). If you swap in a non-pre-quantized base model, set
#    NEEDS_EXPLICIT_QUANT = True so 4-bit quantization still gets applied on load.

NEEDS_EXPLICIT_QUANT = False

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

_MODEL_CACHE = {}  # family -> {"model": PeftModel, "tokenizer": AutoTokenizer, "loaded_adapters": set()}


def get_model_and_tokenizer(family: str, task: str):
    cfg = CONFIG[task][family]
    base_id, adapter_id = cfg["base"], cfg["adapter"]

    if family not in _MODEL_CACHE:
        print(f"[{family}] Loading base model {base_id} ...")
        tok = AutoTokenizer.from_pretrained(base_id)
        if tok.pad_token_id is None:
            tok.pad_token = tok.eos_token
        load_kwargs = dict(device_map="auto")
        if NEEDS_EXPLICIT_QUANT:
            load_kwargs["quantization_config"] = BNB_CONFIG
        base_model = AutoModelForCausalLM.from_pretrained(base_id, **load_kwargs)
        print(f"[{family}] Attaching adapter for '{task}': {adapter_id}")
        model = PeftModel.from_pretrained(base_model, adapter_id, adapter_name=task)
        _MODEL_CACHE[family] = {"model": model, "tokenizer": tok, "loaded_adapters": {task}}
    else:
        entry = _MODEL_CACHE[family]
        model = entry["model"]
        if task not in entry["loaded_adapters"]:
            print(f"[{family}] Attaching adapter for '{task}': {adapter_id}")
            model.load_adapter(adapter_id, adapter_name=task)
            entry["loaded_adapters"].add(task)
        model.set_adapter(task)

    return _MODEL_CACHE[family]["model"], _MODEL_CACHE[family]["tokenizer"]


In [5]:
# 5. Prompt templates + generation helper

def build_prompt(task: str, **kwargs) -> str:
    if task == "qa":
        return (
            "तलको प्रसंग पढेर प्रश्नको उत्तर दिनुहोस्। "
            "यदि प्रसंगमा उत्तर छैन भने 'उत्तर छैन' भन्नुहोस्।\n\n"
            f"प्रसंग: {kwargs['context']}\n\n"
            f"प्रश्न: {kwargs['question']}"
        )
    elif task == "translation":
        return f"Translate the following Nepali sentence into fluent, natural English:\n\n{kwargs['text']}"
    elif task == "summarization":
        return f"तलको लेखको संक्षिप्त र स्पष्ट सारांश नेपाली भाषामा नै लेख्नुहोस्:\n\n{kwargs['text']}"
    raise ValueError(f"Unknown task: {task}")


def generate(family: str, task: str, prompt: str, use_adapter: bool) -> str:
    model, tok = get_model_and_tokenizer(family, task)
    params = GEN_PARAMS[task]

    messages = [{"role": "user", "content": prompt}]
    # Render to text first, then tokenize explicitly — apply_chat_template's return_tensors
    # behavior (plain tensor vs. BatchEncoding) varies across transformers versions, so we
    # avoid that ambiguity entirely by always going through tok(...) which reliably returns
    # a BatchEncoding we can pass to generate() with **enc.
    prompt_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)

    terminators = [tok.eos_token_id]
    eot_id = tok.convert_tokens_to_ids("<|eot_id|>")
    if eot_id is not None and eot_id != tok.unk_token_id and eot_id not in terminators:
        terminators.append(eot_id)

    do_sample = params["temperature"] > 0
    gen_kwargs = dict(
        max_new_tokens=params["max_new_tokens"],
        do_sample=do_sample,
        eos_token_id=terminators,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
        repetition_penalty=1.05,   # mild, helps both models
        no_repeat_ngram_size=3,
    )
    if do_sample:
        gen_kwargs["temperature"] = params["temperature"]
        gen_kwargs["top_p"] = 0.9

    ctx = model.disable_adapter() if not use_adapter else nullcontext()
    with ctx:
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs)
    new_tokens = out[0][enc["input_ids"].shape[-1]:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return text if text else "(empty generation)"


def generate_pair(family: str, task: str, prompt: str):
    """Returns (base_output, finetuned_output) for one family (llama)."""
    base_out = generate(family, task, prompt, use_adapter=False)
    ft_out = generate(family, task, prompt, use_adapter=True)
    return base_out, ft_out


In [6]:
# 6. LLM-as-Judge — scores Base vs. Fine-Tuned outputs
#
#    Provider: Groq (free tier), model = llama-3.3-70b-versatile.
#    Why the switch from Gemini: Google's Gemini API free tier has an ongoing,
#    widely-reported bug in mid-2026 where brand-new/free-tier projects get
#    "403 PERMISSION_DENIED — Your project has been denied access" on every
#    generateContent call, with no fix from Google yet (see
#    discuss.ai.google.dev threads from May-Aug 2026). Groq's free tier has no
#    such reports: 30 RPM / 1,000 RPD on llama-3.3-70b-versatile — for a 90-item
#    question bank (1 judge call per item, comparing base vs. FT together) that's
#    ~90 calls total, well inside quota. Set JUDGE_PROVIDER = "gemini" below if
#    Google fixes the free-tier bug and you'd rather use Gemini instead.

JUDGE_PROVIDER = "groq"   # "groq" (recommended, stable) or "gemini" (currently broken on free tier for many accounts)

import json
import time

JUDGE_SCHEMA_DESC = (
    'Respond with ONLY a JSON object, no other text, matching exactly this shape: '
    '{"winner": "base" | "finetuned" | "tie", "base_score": <integer 1-5>, '
    '"finetuned_score": <integer 1-5>, "reasoning": "<1-2 sentence string>"}'
)

JUDGE_RUBRIC = {
    "qa": (
        "You are evaluating Nepali extractive question-answering. Score each answer 1-5 on: "
        "(a) correctness relative to the context, (b) whether it stays strictly grounded in the "
        "context with no hallucinated facts, and (c) conciseness. If the context genuinely has no "
        "answer, the correct behavior is to say so — reward that."
    ),
    "translation": (
        "You are evaluating Nepali-to-English translation. Score each translation 1-5 on: "
        "(a) faithfulness to the Nepali source meaning, (b) fluency/naturalness of the English, "
        "and (c) completeness (nothing dropped or added)."
    ),
    "summarization": (
        "You are evaluating Nepali summarization. Score each summary 1-5 on: "
        "(a) faithfulness to the source with no hallucinated facts, (b) coverage of the key points, "
        "and (c) whether it reads as a coherent, complete Nepali passage (not truncated mid-sentence)."
    ),
}


def _judge_prompt(task: str, source_fields: dict, base_out: str, ft_out: str) -> str:
    context_block = "\n".join(f"{k}: {v}" for k, v in source_fields.items())
    return (
        f"{JUDGE_RUBRIC[task]}\n\n"
        f"--- INPUT ---\n{context_block}\n\n"
        f"--- OUTPUT A (base model) ---\n{base_out}\n\n"
        f"--- OUTPUT B (fine-tuned model) ---\n{ft_out}\n\n"
        "Compare Output A and Output B. Decide the winner "
        "('base', 'finetuned', or 'tie' if genuinely equal quality), give a 1-5 score for each, "
        f"and a short reasoning (1-2 sentences).\n\n{JUDGE_SCHEMA_DESC}"
    )


def _parse_judge_json(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
    return json.loads(text.strip())


# ---------------- Groq backend ----------------
if JUDGE_PROVIDER == "groq":
    get_ipython().system('pip install -q -U groq')
    from groq import Groq

    GROQ_API_KEY = None
    try:
        from kaggle_secrets import UserSecretsClient
        GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
        print("Loaded GROQ_API_KEY from Kaggle Secrets.")
    except Exception:
        import getpass
        GROQ_API_KEY = getpass.getpass("Enter your Groq API key (from console.groq.com/keys): ")

    groq_client = Groq(api_key=GROQ_API_KEY)
    JUDGE_MODEL_PRIMARY = "llama-3.3-70b-versatile"
    JUDGE_MODEL_FALLBACK = "llama-3.1-8b-instant"  # much higher RPD if 70B's daily cap is hit

    def _call_judge_backend(prompt: str, model: str):
        resp = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        return resp.choices[0].message.content

# ---------------- Gemini backend (kept for when Google fixes the free-tier bug) ----------------
elif JUDGE_PROVIDER == "gemini":
    get_ipython().system('pip install -q -U google-genai')
    from google import genai
    from google.genai import types

    GEMINI_API_KEY = None
    try:
        from kaggle_secrets import UserSecretsClient
        GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
        print("Loaded GEMINI_API_KEY from Kaggle Secrets.")
    except Exception:
        import getpass
        GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key (from aistudio.google.com/apikey): ")

    judge_client = genai.Client(api_key=GEMINI_API_KEY)
    JUDGE_MODEL_PRIMARY = "gemini-2.5-flash"
    JUDGE_MODEL_FALLBACK = "gemini-2.5-flash-lite"

    def _call_judge_backend(prompt: str, model: str):
        resp = judge_client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0),
        )
        return resp.text

else:
    raise ValueError(f"Unknown JUDGE_PROVIDER: {JUDGE_PROVIDER!r}")


def judge_pair(task: str, source_fields: dict, base_out: str, ft_out: str, max_retries: int = 4) -> dict:
    """Calls the configured judge backend to compare base vs. fine-tuned output for one item.
    Retries with backoff on rate-limit errors and falls back to the higher-quota model if the
    primary model's daily quota is exhausted."""
    prompt = _judge_prompt(task, source_fields, base_out, ft_out)
    model = JUDGE_MODEL_PRIMARY
    last_err = None
    for attempt in range(max_retries):
        try:
            raw = _call_judge_backend(prompt, model)
            return _parse_judge_json(raw)
        except Exception as e:
            last_err = e
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg or "rate_limit" in msg.lower():
                if model == JUDGE_MODEL_PRIMARY:
                    print(f"{model} quota/rate limit hit — switching to fallback model.")
                    model = JUDGE_MODEL_FALLBACK
                    continue
                wait = 2 ** attempt
                print(f"Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                time.sleep(1.5)
    return {"winner": "error", "base_score": 0, "finetuned_score": 0,
            "reasoning": f"Judge call failed after retries: {last_err}"}


def format_verdict(v: dict) -> str:
    if v["winner"] == "error":
        return f"Judge error: {v['reasoning']}"
    label = {"base": "Base wins", "finetuned": "Fine-tuned wins", "tie": "Tie"}[v["winner"]]
    return f"{label}  (Base: {v['base_score']}/5, Fine-tuned: {v['finetuned_score']}/5)\n\n{v['reasoning']}"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.0 MB/s eta 0:00:00


StdinNotImplementedError: getpass was called, but this frontend does not support input requests.

In [ ]:
# 7. Task callbacks for the Gradio UI
#    Each returns: llama_base, llama_ft, judge_verdict

def run_qa(context, question):
    if not context.strip() or not question.strip():
        msg = "Please provide both context and a question."
        return msg, msg, ""
    prompt = build_prompt("qa", context=context, question=question)
    llama_base, llama_ft = generate_pair("llama", "qa", prompt)
    verdict = judge_pair("qa", {"Context": context, "Question": question}, llama_base, llama_ft)
    return llama_base, llama_ft, format_verdict(verdict)


def run_translation(text):
    if not text.strip():
        msg = "Please provide a Nepali sentence or paragraph to translate."
        return msg, msg, ""
    prompt = build_prompt("translation", text=text)
    llama_base, llama_ft = generate_pair("llama", "translation", prompt)
    verdict = judge_pair("translation", {"Nepali Text": text}, llama_base, llama_ft)
    return llama_base, llama_ft, format_verdict(verdict)


def run_summarization(text):
    if not text.strip():
        msg = "Please provide a Nepali passage to summarize."
        return msg, msg, ""
    prompt = build_prompt("summarization", text=text)
    llama_base, llama_ft = generate_pair("llama", "summarization", prompt)
    verdict = judge_pair("summarization", {"Source Passage": text}, llama_base, llama_ft)
    return llama_base, llama_ft, format_verdict(verdict)


In [ ]:
# 8. Gradio UI — one page, task tabs, Llama-3.2-3B outputs + Judge verdict shown side by side

def task_tab(task_key, run_fn, input_components_fn, title):
    """Builds one tab: inputs, run button, Llama base/fine-tuned outputs, judge verdict."""
    with gr.TabItem(title):
        inputs = input_components_fn()
        run_btn = gr.Button(f"Run {title} Comparison", variant="primary")

        gr.Markdown("## Model Outputs")
        gr.Markdown("### Llama-3.2-3B")
        with gr.Row():
            llama_base = gr.Textbox(label="Llama-3.2-3B — Base Output", interactive=False, lines=3)
            llama_ft = gr.Textbox(label="Llama-3.2-3B — Fine-Tuned Output", interactive=False, lines=3)

        gr.Markdown("### LLM-as-Judge Verdict (Gemini)")
        verdict_box = gr.Textbox(label="Verdict", interactive=False, lines=3)

        run_btn.click(
            run_fn,
            inputs=inputs,
            outputs=[llama_base, llama_ft, verdict_box],
        )


with gr.Blocks(title="Nepali LLM: Llama-3.2-3B Comparator") as demo:
    gr.Markdown(
        "# Nepali LLM Evaluation Dashboard — Llama-3.2-3B\n"
        "Base vs. Fine-Tuned — pick a task tab, enter input, and click Run. "
        "Gemini automatically judges which output is better."
    )

    with gr.Tabs():
        task_tab(
            "qa", run_qa,
            lambda: [
                gr.Textbox(label="Context (Nepali)", lines=4, placeholder="प्रसंग यहाँ लेख्नुहोस्..."),
                gr.Textbox(label="Question (Nepali)", placeholder="प्रश्न यहाँ लेख्नुहोस्..."),
            ],
            "Question Answering",
        )
        task_tab(
            "translation", run_translation,
            lambda: [gr.Textbox(label="Nepali Text", lines=4, placeholder="अनुवाद गर्नुपर्ने नेपाली वाक्य वा अनुच्छेद यहाँ लेख्नुहोस्...")],
            "Translation (Nepali → English)",
        )
        task_tab(
            "summarization", run_summarization,
            lambda: [gr.Textbox(label="Nepali Source Passage", lines=6, placeholder="सारांश गर्नुपर्ने नेपाली लेख यहाँ राख्नुहोस्...")],
            "Summarization (Nepali → Nepali)",
        )


In [ ]:
# 10. Launch — share=True gives a public gradio.live link, which is what you need on Kaggle
demo.queue().launch(share=True, debug=True)


## 11. Batch Evaluation - run the full 90-item question bank

Runs generate + judge over a file of evaluation items (30 QA + 30 translation + 30 summarization, or however your 90-item bank is split) instead of clicking through the UI one by one. Produces a CSV with per-item verdicts and a win-rate summary.

In [ ]:
# 11. Batch Evaluation — run the full 90-item question bank through generate + judge
#
#    Expects a JSON-Lines file, one object per line, e.g.:
#      {"task": "qa", "context": "...", "question": "..."}
#      {"task": "translation", "text": "..."}
#      {"task": "summarization", "text": "..."}
#    Update QUESTION_BANK_PATH to point at your uploaded 90-item file.
#
#    Cost check: 90 items x 1 judge call each (base+FT compared together in one call)
#    = 90 Gemini calls. gemini-2.5-flash's free-tier daily quota (~250 RPD as of mid-2026)
#    comfortably covers this in a single run; the sleep below paces calls under the ~10 RPM
#    limit too (generation itself runs locally on the GPU, only judging hits the API).

import pandas as pd

QUESTION_BANK_PATH = "/kaggle/input/your-dataset/question_bank.jsonl"  # <-- update this


def load_question_bank(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def run_batch_evaluation(path, sleep_between=6.5):
    items = load_question_bank(path)
    results = []
    for i, item in enumerate(items):
        task = item["task"]
        if task == "qa":
            prompt = build_prompt("qa", context=item["context"], question=item["question"])
            source_fields = {"Context": item["context"], "Question": item["question"]}
        else:
            prompt = build_prompt(task, text=item["text"])
            source_fields = {"Text": item["text"]}

        base_out, ft_out = generate_pair("llama", task, prompt)
        verdict = judge_pair(task, source_fields, base_out, ft_out)

        results.append({
            "index": i, "task": task, **source_fields,
            "base_output": base_out, "finetuned_output": ft_out,
            "winner": verdict["winner"], "base_score": verdict["base_score"],
            "finetuned_score": verdict["finetuned_score"], "reasoning": verdict["reasoning"],
        })
        print(f"[{i+1}/{len(items)}] task={task} winner={verdict['winner']}")
        time.sleep(sleep_between)

    df = pd.DataFrame(results)
    df.to_csv("/kaggle/working/judge_results.csv", index=False)
    print("\n=== Win-rate summary ===")
    print(df.groupby("task")["winner"].value_counts(normalize=True).round(2))
    return df

# Uncomment once QUESTION_BANK_PATH points at your uploaded 90-item file:
# results_df = run_batch_evaluation(QUESTION_BANK_PATH)
